In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/modules/python-utils")
sys.path.append("/workspaces/dev/modules/ai-utils")

In [ ]:
import librosa
import numpy as np
from pathlib import Path
from IPython.display import Audio

In [ ]:
SEED = 42
SAMPLE_RATE = 16000
HYPERPARAMETER = "/workspaces/dev/hyperparameters/libri/sentence_error_47_4.yml"
# SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test/test-clean/61/70968/61-70968-0001.flac"
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/test/20080901/023_001_EN_Mladenov/en.OS.man-diar.mp4"
LOG_FILE_PATH = "/workspaces/dev/logs/core.log"

In [ ]:
src = Path(SOURCE)
src.exists()
log = Path(LOG_FILE_PATH)
if log.exists():
    with log.open("w"): pass

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, TokenState
from sj_utils.audio_utils import segment_audio, load_audio_from_mp4

In [ ]:
np.random.seed(SEED)
audio, sr = load_audio_from_mp4(src, sr=SAMPLE_RATE)
# audio, sr = librosa.load(src, sr=SAMPLE_RATE)
segments = segment_audio(audio)
Audio(audio, rate=sr)

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter=HYPERPARAMETER)

In [ ]:
param = Param()
completed = []
for i, segment in enumerate(segments):
    param.chunk = segment
    ctx:TokenState = token_streamer.process(param, get_context = True)
    result = ctx.extract()
    completed.extend(result.completed)
    param.update(result, update_prompt=True)

    # print(f"Processed segment {i+1}")
    # print(f"\t")

In [ ]:
completed.extend(result.candidate)

In [ ]:
for s in completed:
    print(s.lang, s.text)